# Next step: step 12b, local pools
**What it adds:** running models in parallel on one machine.

- **In Python:** create a pool once and pass its `map`. `simr.available_cpus()` is the CPUs this process may use (under
  Slurm, the job's), and `simr.configure_logging` sets up logging with the process name in each line, for the main
  process and, as the pool's initializer, each worker.
- **On the command line:** `--mode local` and `--workers N` (default: `simr.available_cpus()`).
- **While results arrive,** the statuses so far go into the catalog at most once a minute (`simr.CATALOG_WRITE_SECONDS`)
  and at the end; cells not yet run are `null`.
- **A worker that dies** (e.g. a crash in SPAM's Fortran) breaks the pool: the unfinished models are logged, the catalog
  keeps what finished with `"finished": null`, and `simr.SimrRunError` is raised (exit code 1 on the command line). A
  rerun reuses everything saved.
- Results are identical with any number of workers: each entry's seed is in its config.

## What changed
**`src/galactic_tidal_constraints/simr/simr.py`.** Collecting results now writes the catalog as it goes, and a broken
pool is caught:
```python
try:
    for task_rows, outcome in zip(rows, map(_run_model, tasks)):
        ...                                                   # fill the arrays, as in 12a
        if catalog is not None and time.monotonic() - last_write >= CATALOG_WRITE_SECONDS:
            _record_status(catalog_file, catalog, run_record, status, seconds, errors)
            last_write = time.monotonic()
except BrokenExecutor as error:                               # a worker died: nothing more will come back
    logger.error("a worker died (%s); %d of %d models unfinished: %s", ...)
    _record_status(...)                                       # what finished, with "finished" still None
    raise SimrRunError(f"a worker died with {len(unfinished)} models unfinished; rerun to resume") from error
```
New helpers: `configure_logging(level)`, `available_cpus()`, `SimrRunError`.

**`src/galactic_tidal_constraints/simr/__main__.py`.** `--mode local` creates the pool once and closes it at the end.
Workers start fresh ("spawn"), as MPI's do:
```python
with ProcessPoolExecutor(max_workers=workers, mp_context=multiprocessing.get_context("spawn"),
                         initializer=simr.configure_logging, initargs=(args.log_level,)) as pool:
    return simr.run(models, request, output=args.output, map=pool.map)
```
**Tests:** section "Step 12b" in `tests/test_simr.py` (a pool gives the same results as one process; statuses written
while running; a broken pool, simulated and real) and `tests/test_cli.py`. **Docs:** `docs/design/simr.md`, Spreading
the work and The command line.

In [ ]:
import json
import multiprocessing
import subprocess
import sys
import tempfile
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np

from galactic_tidal_constraints.simr import simr

base = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                 12.97415, 135.11902, 97.54035, 305.01225])
models = np.stack([base * (1 + 0.01 * k) for k in range(8)])   # 8 slightly different models
request = {"run": {"return": ["sim.particles"]}, "simulator": {"n1": 1500, "n2": 1500}}
work = Path(tempfile.mkdtemp(prefix="simr_12b_"))
print("CPUs this process may use:", simr.available_cpus())

## 1. A pool in Python
One process, then a pool of 4. The particles are identical:

In [ ]:
start = time.perf_counter()
solo = simr.run(models, request)
print(f"1 process: {time.perf_counter() - start:.1f} s")

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4, mp_context=multiprocessing.get_context("spawn")) as pool:
    pooled = simr.run(models, request, map=pool.map)
print(f"4 workers: {time.perf_counter() - start:.1f} s")

print("identical:", all(np.array_equal(solo.data["sim.particles"][m, 0], pooled.data["sim.particles"][m, 0])
                        for m in range(len(models))))

## 2. On the command line
Each log line names its process; the failures and the count come from the workers and the main process:

In [ ]:
(work / "models.txt").write_text("\n".join(",".join(str(v) for v in model) for model in models) + "\n")
(work / "config.json").write_text(json.dumps({"run": {"save": ["sim.orbit"]}, "simulator": {"n1": 500, "n2": 500}}))
done = subprocess.run([sys.executable, "-m", "galactic_tidal_constraints.simr", "--models", str(work / "models.txt"),
                       "--config", str(work / "config.json"), "--output", str(work / "runs"),
                       "--mode", "local", "--workers", "4"], capture_output=True, text=True)
print(done.stderr.strip())
print("exit code:", done.returncode)

## 3. A worker that dies
A crash in SPAM's Fortran kills its process outright, which Python can't catch. Here a stand-in does the same
(`os._exit`) on the 4th model. The run stops, and the catalog keeps what finished; a rerun reuses it. ("fork" workers
are used here only so they inherit the stand-in.)

In [ ]:
import os

real_orbit = simr.SIM_DATA["sim.orbit"]


def crash_on_the_fourth(params, config):
    if np.array_equal(params, models[3]):
        os._exit(1)
    return real_orbit(params, config)


simr.SIM_DATA["sim.orbit"] = crash_on_the_fourth
save_orbits = {"run": {"save": ["sim.orbit"]}, "simulator": {"n1": 500, "n2": 500}}
try:
    with ProcessPoolExecutor(max_workers=1, mp_context=multiprocessing.get_context("fork")) as pool:
        simr.run(models, save_orbits, output=work / "crash", map=pool.map)
except simr.SimrRunError as error:
    print("SimrRunError:", error)
finally:
    simr.SIM_DATA["sim.orbit"] = real_orbit

record = simr.load_catalog(work / "crash")["runs"][0]
print("finished:", record["finished"], "  status:", [row[0] for row in record["status"]["sim"]])
again = simr.run(models, save_orbits, output=work / "crash")
print("rerun:", again.status["sim"][:, 0].tolist())